# DWH Performance Patterns

## Overview
Data warehouse queries scan millions of rows. Performance comes from three complementary strategies: eliminating rows before scanning (partitioning), pre-computing expensive aggregations (materialized views), and writing queries the planner can optimise (sargable predicates, early filtering).

**Performance hierarchy (biggest impact first):**
1. **Partitioning** — query hits 1 partition instead of 5 years of data
2. **Materialized views** — query reads a 1,000-row summary instead of 100M-row fact table
3. **Indexes** — BRIN on date ranges; B-tree on dimension keys
4. **Query structure** — filter early, avoid non-sargable predicates, aggregate before joining dimensions

---

In [ ]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


---
## Table partitioning

In [ ]:
print("=== Table partitioning for large fact tables ===")
print()

print("Why partition fact tables?")
print("""
  fact_policy with 5 years of data = 100M rows
  Query: WHERE date_key BETWEEN 20240101 AND 20241231
  Without partitioning: full table scan or index scan on 100M rows
  With range partitioning on date_key: query hits only the 2024 partition (~20M rows)
  → 5x fewer rows scanned
""")

partition_types = [
    ("RANGE",  "Partition by a range of values (most common for DWH)",
     "date_key ranges: one partition per year/quarter/month"),
    ("LIST",   "Partition by explicit list of values",
     "province: one partition per province; product_line"),
    ("HASH",   "Partition by hash of a column value",
     "Evenly distribute rows with no natural range/list key"),
]
print("PostgreSQL partition types:")
for typ, desc, example in partition_types:
    print(f"  {typ:<6s}  {desc}")
    print(f"         Example: {example}")
    print()

print("Creating a range-partitioned fact table (PostgreSQL):")
print("""
  CREATE TABLE fact_policy (
      policy_key      BIGINT,
      date_key        INTEGER NOT NULL,  -- partition key
      customer_key    INTEGER,
      product_key     INTEGER,
      agent_key       INTEGER,
      premium_amount  NUMERIC(12,2),
      claim_amount    NUMERIC(12,2)
  ) PARTITION BY RANGE (date_key);

  -- Create partitions for each year:
  CREATE TABLE fact_policy_2022 PARTITION OF fact_policy
      FOR VALUES FROM (20220101) TO (20230101);
  CREATE TABLE fact_policy_2023 PARTITION OF fact_policy
      FOR VALUES FROM (20230101) TO (20240101);
  CREATE TABLE fact_policy_2024 PARTITION OF fact_policy
      FOR VALUES FROM (20240101) TO (20250101);

  -- Indexes are created per partition:
  CREATE INDEX ON fact_policy_2024 USING BRIN (date_key);
  CREATE INDEX ON fact_policy_2024 (customer_key);

  -- Partition pruning: PostgreSQL automatically queries only relevant partition:
  EXPLAIN SELECT * FROM fact_policy WHERE date_key = 20241001;
  -- -> Seq Scan on fact_policy_2024 (not fact_policy_2022 or _2023)
""")


---
## Materialized views

In [ ]:
print("=== Materialized views for pre-aggregated results ===")
print()

# Simulate materialized view with a real table (SQLite doesn't have them)
conn.execute("""
    CREATE TABLE IF NOT EXISTS mv_quarterly_summary AS
    SELECT dd.year, dd.quarter,
           dp.product_line,
           da.region,
           SUM(fp.premium_amount)  AS total_premium,
           SUM(fp.claim_amount)    AS total_claims,
           COUNT(fp.policy_count)  AS n_policies,
           ROUND(100.0 * SUM(fp.claim_amount)
                       / NULLIF(SUM(fp.premium_amount),0), 2) AS loss_ratio_pct
    FROM   fact_policy fp
    JOIN   dim_date    dd ON dd.date_key    = fp.date_key
    JOIN   dim_product dp ON dp.product_key = fp.product_key
    JOIN   dim_agent   da ON da.agent_key   = fp.agent_key
    GROUP  BY dd.year, dd.quarter, dp.product_line, da.region
""")
conn.commit()

rows = conn.execute("""
    SELECT * FROM mv_quarterly_summary
    ORDER BY year, quarter, product_line
""").fetchall()
print("mv_quarterly_summary (pre-aggregated, no joins needed at query time):")
print(f"  {'year':<6s}  {'Q':<3s}  {'product_line':<10s}  {'region':<8s}  {'premium':>10s}  {'loss%':>6s}")
print("  " + "-"*52)
for r in rows:
    print(f"  {r['year']:<6d}  {r['quarter']:<3d}  {r['product_line']:<10s}  "
          f"{r['region']:<8s}  ${r['total_premium']:>9,.0f}  {r['loss_ratio_pct'] or 0:>5.1f}%")

print()
print("PostgreSQL materialized view:")
print("""
  -- Create:
  CREATE MATERIALIZED VIEW mv_quarterly_summary AS
  SELECT dd.year, dd.quarter, dp.product_line, da.region,
         SUM(fp.premium_amount) AS total_premium,
         SUM(fp.claim_amount)   AS total_claims,
         ROUND(100.0 * SUM(fp.claim_amount) / NULLIF(SUM(fp.premium_amount),0), 2) AS loss_ratio_pct
  FROM   fact_policy fp
  JOIN   dim_date dd ON dd.date_key = fp.date_key
  JOIN   dim_product dp ON dp.product_key = fp.product_key
  JOIN   dim_agent da ON da.agent_key = fp.agent_key
  GROUP  BY dd.year, dd.quarter, dp.product_line, da.region
  WITH DATA;                           -- populate immediately

  -- Index on materialized view:
  CREATE INDEX ON mv_quarterly_summary (year, quarter);

  -- Refresh after each ETL load:
  REFRESH MATERIALIZED VIEW CONCURRENTLY mv_quarterly_summary;
  -- CONCURRENTLY: allows reads during refresh (requires unique index)

  -- Drop and recreate (simpler, but locks out reads):
  REFRESH MATERIALIZED VIEW mv_quarterly_summary;
""")


---
## Query optimisation patterns

In [ ]:
print("=== Query optimisation patterns for DWH ===")
print()

optimisations = [
    ("Filter early with CTEs / subqueries",
     "Apply WHERE before JOINs to reduce row count flowing through the query",
     """WITH recent AS (SELECT * FROM fact_policy WHERE date_key >= 20240101)
SELECT r.*, c.full_name FROM recent r JOIN dim_customer c ON c.customer_key = r.customer_key;"""),
    ("Avoid SELECT * from fact tables",
     "Wide fact tables have many columns; SELECT only needed measures",
     "SELECT date_key, SUM(premium_amount) FROM fact_policy GROUP BY date_key;"),
    ("Use integer date_key not DATE join",
     "Joining on INTEGER date_key is faster than BETWEEN/CAST on a DATE column",
     "WHERE date_key BETWEEN 20240101 AND 20241231  -- not WHERE full_date::DATE"),
    ("Avoid non-sargable WHERE on dimension attributes",
     "UPPER(province) = 'ON' prevents index use; store normalised data instead",
     "WHERE province = 'ON'  -- store values normalised at load time"),
    ("Partial aggregation in CTE before joining dimensions",
     "Aggregate fact table first, then join to dimensions for labels",
     """WITH agg AS (SELECT product_key, SUM(premium) AS total FROM fact_policy GROUP BY product_key)
SELECT dp.product_name, agg.total FROM agg JOIN dim_product dp ON dp.product_key = agg.product_key;"""),
]

for name, principle, example in optimisations:
    print(f"  ▸ {name}")
    print(f"    {principle}")
    print(f"    Example:")
    for line in example.split('\n'):
        print(f"      {line}")
    print()


---
## Common Pitfalls

**1. Partitioning on a low-cardinality column**
Partitioning `fact_policy` by `province` (10 values) creates 10 partitions, each scanned for queries spanning multiple provinces. Partition by time (date_key range) for DWH fact tables — time filters are almost always present in analytical queries.

**2. Forgetting to add indexes to each partition**
In PostgreSQL declarative partitioning, `CREATE INDEX ON fact_policy (customer_key)` creates indexes on all existing partitions but NOT on future partitions. Add indexes to each new partition as it is created, or use a DDL automation script.

**3. Materialised view not refreshed after ETL load**
A materialized view of quarterly summaries becomes stale immediately after the nightly ETL load. Queries return last night's data until `REFRESH MATERIALIZED VIEW` is run. Automate refresh as the final step in the ETL pipeline.

**4. Non-sargable WHERE defeating partition pruning**
`WHERE EXTRACT(YEAR FROM full_date) = 2024` on a date-range-partitioned table may not trigger partition pruning if PostgreSQL can't resolve the predicate at plan time. Use `WHERE date_key BETWEEN 20240101 AND 20241231` with the integer partition key.

**5. Joining to the full dim_date instead of filtering first**
A query that joins `fact_policy` to all 1096 rows of `dim_date` just to filter `WHERE year = 2024` is wasteful. Filter on `date_key BETWEEN 20240101 AND 20241231` directly on the fact table before or instead of the join.

**6. Using DISTINCT to fix a broken join instead of fixing the join**
`SELECT DISTINCT customer_id, SUM(premium) FROM fact JOIN dim ...` — if DISTINCT is needed to avoid duplicates, the JOIN is probably a many-to-many and a GROUP BY is needed. DISTINCT on a wide query is slow; GROUP BY with the correct grain is both correct and faster.


---
*sql_methods_library - Samantha McGarrigle*